# 在 Google Colab 上跑 RFD4-Proteina（Tesla T4 专用）

**这个文件只给 T4 用。** 有 L4 / A100 额度请用 `RFD4-colab.ipynb`——那边不用改源码、跑 bf16，快得多。

T4 是 Turing **sm_75**，没有原生 bf16 代码生成。而 RFD4 默认 `precision: bf16-mixed`，`norm.py` 里还有一处无条件 `torch.compile(..., fullgraph=True)`，于是会依次撞上：

```
RuntimeError: Invalid backend                                        # SDPA 调度
RuntimeError: torch.compile with fullgraph=True found no compiled frames
  - _rms_norm_preserve_dtype: Dynamo decided to skip the frame: BF16 is not supported
```

**已验证的绕法**（本文流程，实测 `monomer-short` 30 秒出结构）：

1. 把 `norm.py` 那行 `torch.compile` 退回 eager；
2. 把 `pair_bias_attn.py` 的 SDPA 强制走 einsum 分支；
3. 全程 `precision=32-true` 跑 fp32。

代价：慢、显存吃紧，15GB 只适合小例子。环境准备部分（清变量、clone、setup）与主文件完全一致。

In [ ]:
!python -V
!nvidia-smi
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
%cd /content
!rm -rf RFD4-Proteina
# --depth 1：这个仓库历史里有大量 notebook 输出，全量 clone 超过 100MB
!git clone --depth 1 https://github.com/RosettaCommons/RFD4-Proteina.git
%cd /content/RFD4-Proteina
!pwd && ls

## 为什么 `make setup` 在 Colab 上不灵

1. `make setup` 等价于 `bash setup.sh`，它调用 `scripts/setup_venv.sh`。该脚本先 `uv venv` 建 `.venv`，再 `uv pip install uv==0.13.4`，然后**用 `.venv/bin/uv`** 装依赖（为了拿到支持 `--torch-backend cu132` 的 uv 版本）。
2. `uv venv` 生成的 venv 里**本来就没有 `bin/uv` 和 `bin/pip`**：uv 默认不往 venv 里装 pip，`uv` 是后来那条 `uv pip install uv` 装进去的。
3. 但 Colab 给整个会话预设了一批指向它自己 `/usr` 环境的变量，于是那条命令打印的是 `Using Python 3.12.13 environment at: /usr`，包装进了系统环境——`.venv/bin/uv` 永远不会出现，脚本第 81 行 `No such file or directory`，`make setup` 以 127 退出。

要清的清单（下面那个 cell 一次清完并打印结果）：

| 变量 | Colab 的值 | 不清会怎样 |
| --- | --- | --- |
| `UV_SYSTEM_PYTHON` | `true` | uv 无视 `.venv`，装进 `/usr`，`make setup` 报 127 |
| `UV_CONSTRAINT` / `UV_BUILD_CONSTRAINT` | 空 | 把依赖钉到 Colab 自带版本 |
| `UV_INSTALL_DIR` | `/usr/local/bin` | `uv` 被装到 `/usr/local/bin` |
| `PYTHONPATH` | `/env/python` | `.venv` 里的 python 也去 import `/usr` 的包 |
| `MPLBACKEND` | `module://matplotlib_inline.backend_inline` | `.venv` 里没有 `matplotlib_inline`，`import matplotlib` 直接 `ValueError: Key backend`，进而拖垮 `lightning` / `rfd4-generate` |

In [2]:
import os

# Colab 给整个会话预设了一批变量，它们全都指向 Colab 自己的 /usr 环境：
#   UV_SYSTEM_PYTHON=true      -> uv 无视 .venv，直接装进 /usr（make setup 失败的直接原因）
#   UV_CONSTRAINT / UV_BUILD_CONSTRAINT -> 把你的依赖钉到 Colab 自带版本
#   UV_INSTALL_DIR=/usr/local/bin       -> uv 把自己装到 /usr/local/bin
#   PYTHONPATH=/env/python     -> 让 .venv 里的 python 也去 import /usr 的包
#   MPLBACKEND=module://matplotlib_inline.backend_inline
#                              -> .venv 里没有 matplotlib_inline，
#                                 import matplotlib 会 ValueError: Key backend
COLAB_VARS = [
    'UV_SYSTEM_PYTHON',
    'UV_CONSTRAINT',
    'UV_BUILD_CONSTRAINT',
    'UV_INSTALL_DIR',
    'UV_PROJECT_ENVIRONMENT',
    'UV_INDEX_URL',
    'PYTHONPATH',
    'MPLBACKEND',
]

extra_uv = [k for k in os.environ if k.startswith('UV_') and k not in COLAB_VARS]
for key in COLAB_VARS + extra_uv:
    if key in os.environ:
        print(f'unset {key}={os.environ.pop(key)}')

print()
print('剩余 UV_*    :', [k for k in os.environ if k.startswith('UV_')])
print('MPLBACKEND  :', os.environ.get('MPLBACKEND'))
print('PYTHONPATH  :', os.environ.get('PYTHONPATH'))

unset UV_SYSTEM_PYTHON=true
unset UV_CONSTRAINT=
unset UV_BUILD_CONSTRAINT=
unset UV_INSTALL_DIR=/usr/local/bin
unset PYTHONPATH=/env/python
unset MPLBACKEND=module://matplotlib_inline.backend_inline

剩余 UV_*    : []
MPLBACKEND  : None
PYTHONPATH  : None


## 可选：配置 `HF_TOKEN`

不配也能跑——RFD4 的模型仓库是公开的，匿名下载只是**限速**，日志里会有这条警告：

```
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN
```

> **VS Code 的 Colab 扩展没有 Secrets 面板。** 它的命令只有 `Remove Server` / `Sign Out` / `Mount Google Drive to Server...`，所以 `google.colab.userdata` 在扩展里读不到东西。Secret 必须在某个 cell 里“进场”——但**不等于必须把 token 明文写进 notebook**。

四条通道，按“notebook 里是否出现 token 明文”排序：

| 通道 | notebook 里有什么 | 每次换实例要做什么 |
| --- | --- | --- |
| **1. 不配** | 什么都没有 | 无。只是限速 |
| **2. Google Drive 文件**（推荐给 VS Code） | 只有一个路径 | 挂 Drive 后自动读，无感 |
| **3. `.env.local`** | `MANUAL_TOKEN = "..."` 或由 Drive 读出 | 换实例 `/content` 清空，要重跑这个 cell |
| **4. Colab Secrets** | 无（仅网页端） | 无 |

**通道 2 怎么用**（token 永远不进 notebook）：

1. 在 <https://drive.google.com> 上传一个文本文件，比如 `MyDrive/rfd4/hf_token.txt`，内容只有一行 `hf_xxx`；
2. 下面 cell 里设 `TOKEN_FILE = "/content/drive/MyDrive/rfd4/hf_token.txt"`；
3. 先挂 Drive（扩展里：命令面板 → `Colab: Mount Google Drive to Server...`；网页端就是 `drive.mount`），再跑这个 cell。

拿 token：<https://huggingface.co/settings/tokens>（read 权限即可）。

In [ ]:
import os
from pathlib import Path

# 可选：Hugging Face token。留空/跳过都行，只是 HF 下载会限速。
#
# 优先级：Colab Secrets（仅网页端） > TOKEN_FILE（推荐，token 不进 notebook） > MANUAL_TOKEN > 内核环境变量

# ① Drive 里的 token 文件：内容一行 hf_xxx。需要先挂 Drive。
#    VS Code 扩展：命令面板 → "Colab: Mount Google Drive to Server..."
TOKEN_FILE = ""

# ② 手填：会留在 ipynb 里，别提交/分享
MANUAL_TOKEN = ""

# True：把 token 写进仓库根的 .env.local（gitignore），setup.sh 和之后的 rfd4-generate 都能读到
WRITE_ENV_LOCAL = True
REPO_ROOT = "/content/RFD4-Proteina"

token = None
try:
    from google.colab import userdata          # 仅网页端可用；VS Code 扩展里会抛异常
    token = userdata.get("HF_TOKEN")
except Exception as exc:
    print(f"读不到 Colab Secrets（{type(exc).__name__}），继续回退（VS Code 扩展里这是正常现象）")

if not token and TOKEN_FILE:
    p = Path(TOKEN_FILE)
    token = p.read_text(encoding="utf-8").strip() if p.is_file() else None
    print(f"从 {TOKEN_FILE} 读到 token" if token else f"{TOKEN_FILE} 不存在（Drive 挂了吗？）")

token = (token or MANUAL_TOKEN or os.environ.get("HF_TOKEN") or "").strip()

if not token:
    print("未配置 HF_TOKEN —— 可以直接继续，只是 HF 下载会限速、日志里会有那条 Warning。")
else:
    os.environ["HF_TOKEN"] = token
    # 不打印 token 本身，只给长度和结尾，便于确认「确实读到了」
    print(f"HF_TOKEN 已设置：长度 {len(token)}，结尾 ...{token[-4:]}")

    if WRITE_ENV_LOCAL and not Path(REPO_ROOT).is_dir():
        print(f"{REPO_ROOT} 不存在（还没 clone？），跳过写入 .env.local。")
    elif WRITE_ENV_LOCAL:
        env_file = Path(REPO_ROOT) / ".env.local"
        lines = env_file.read_text(encoding="utf-8").splitlines() if env_file.exists() else []
        kept = [ln for ln in lines if not ln.strip().startswith("HF_TOKEN=")]
        kept.append(f"HF_TOKEN={token}")
        env_file.write_text("\n".join(kept).rstrip() + "\n", encoding="utf-8")
        env_file.chmod(0o600)
        print(f"已写入 {env_file}（gitignore，仅当前实例有效）；setup.sh 的 hf download 会用上它。")
    else:
        print("只设在当前内核环境里；setup.sh 的子进程同样能继承到。")

## 安装（约 7GB 依赖 + 1.7GB 权重，5~15 分钟）

这一步会：下载 uv 托管的 CPython 3.12.10 → 建 `.venv`（Colab 自带的 3.12.13 超出仓库要求的 `<=3.12.10`）→ 把 uv 装进 `.venv` → 装约 7GB 依赖（`torch 2.14.1+cu132`）→ 从 Hugging Face 下载约 1.7GB 权重和 63MB 基准数据。

`--skip-search-envs` 是必须的：`make setup` 会交互式问要不要装 AlphaFold 2 / OpenFold3 环境，Colab 没有 TTY，会卡住。需要时以后再跑 `bash scripts/setup_search_envs.sh`。

In [ ]:
%cd /content/RFD4-Proteina
!bash setup.sh --skip-search-envs

In [3]:
%%bash
cd /content/RFD4-Proteina
set -e

echo '--- python / uv 都在 .venv 里 ---'
.venv/bin/uv --version
.venv/bin/python -c 'import sys; print(sys.version.split()[0], sys.prefix)'

echo '--- 硬件与导入链自查：任何一项失败都会单独报出来，不会中途断掉 ---'
.venv/bin/python - <<'PY'
import importlib
import os

def show(label, mod):
    try:
        m = importlib.import_module(mod)
        print(f'{label:12s}', getattr(m, '__version__', '(no __version__)'), m.__file__)
        return m
    except Exception as exc:
        print(f'{label:12s}', 'IMPORT FAILED ->', type(exc).__name__, exc)
        return None

print('MPLBACKEND  ', os.environ.get('MPLBACKEND'))
print('PYTHONPATH  ', os.environ.get('PYTHONPATH'))
torch = show('torch', 'torch')
show('matplotlib', 'matplotlib')
show('lightning', 'lightning')
show('torchmetrics', 'torchmetrics')
show('rfproteina', 'rfproteina')
if torch is not None and torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('device      ', torch.cuda.get_device_name(0), '| capability', cap)
    print('bf16        ', torch.cuda.is_bf16_supported())
    print('cudnn       ', torch.backends.cudnn.version())
    if cap[0] >= 8:
        print()
        print(f'  !! sm_{cap[0]}.{cap[1]} 有原生 bf16，不要用本文件的补丁和 precision=32-true。')
        print('    请改用 RFD4-colab.ipynb（默认 bf16 路径，更快）。')
PY
echo '--- 依赖一致性 ---'
.venv/bin/uv pip check --python .venv/bin/python

echo '--- 系统环境里是否还留着旧安装（torchvision::nms 报错就出在这里）---'
if command -v rfd4-generate >/dev/null 2>&1; then
    echo "警告：PATH 上还有 $(command -v rfd4-generate)；一律用 .venv/bin/rfd4-generate"
else
    echo "PATH 上没有其它 rfd4-generate，干净"
fi
ls -l /usr/local/bin/rfd4-generate 2>/dev/null || echo "/usr/local/bin/rfd4-generate 不存在"

echo '--- CLI ---'
.venv/bin/rfd4-generate --help | head -20

--- python / uv 都在 .venv 里 ---
uv 0.13.4 (x86_64-unknown-linux-gnu)
3.12.10 /content/RFD4-Proteina/.venv
--- 硬件与导入链自查：任何一项失败都会单独报出来，不会中途断掉 ---
MPLBACKEND   None
PYTHONPATH   None
torch        2.14.1+cu132 /content/RFD4-Proteina/.venv/lib/python3.12/site-packages/torch/__init__.py
matplotlib   3.11.2 /content/RFD4-Proteina/.venv/lib/python3.12/site-packages/matplotlib/__init__.py
lightning    2.6.6 /content/RFD4-Proteina/.venv/lib/python3.12/site-packages/lightning/__init__.py
torchmetrics 1.9.0 /content/RFD4-Proteina/.venv/lib/python3.12/site-packages/torchmetrics/__init__.py
rfproteina   (no __version__) /content/RFD4-Proteina/src/rfproteina/__init__.py
device       Tesla T4 | capability (7, 5)
bf16         True
cudnn        92400
--- 依赖一致性 ---
--- 系统环境里是否还留着旧安装（torchvision::nms 报错就出在这里）---
警告：PATH 上还有 /usr/local/bin/rfd4-generate；一律用 .venv/bin/rfd4-generate
-rwxr-xr-x 1 root root 314 Nov 17 08:59 /usr/local/bin/rfd4-generate
--- CLI ---
generate is powered by Hydra.

== Configuration

/content/RFD4-Proteina/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Checked 217 packages in 4ms
All installed packages are compatible
Environment variabl

## T4 / sm_75 上的临时绕法（**L4、A100 上不要跑这个 cell**）

**T4 实测：能跑通。** 打补丁 + `precision=32-true` 之后，`smoke-t4` 30 秒完成：

```
sizzling... 🔥  1/1 0:00:30
[rfproteina.generate][INFO] [rank: 0] ✓ Generation complete! Results saved to ./runs/smoke-t4
```

所以 T4 可以留在 notebook 里当“没额度时的备选”。但它跑的是 **fp32 + eager einsum**，比 L4 上的 bf16 慢很多、显存也吃紧，15GB 只够跑小规模例子。

**别再往 SDPA 上找原因**：实测 T4 上 `F.scaled_dot_product_attention` 的 fp32 / bf16 / fp16 × 有 mask / 无 mask 六种组合**全部 OK**，`capability (7,5)`、四个 SDPA flag 全开——`Invalid backend` 不是“缺注意力内核”。

真正的坎是 bf16：T4 没有原生 bf16 inductor 代码生成，而 `norm.py` 在模块加载时无条件编译了一版 RMSNorm：

```python
_compiled_rms_norm = torch.compile(_rms_norm_preserve_dtype, fullgraph=True, dynamic=True)
```

`fullgraph=True` 不留降级余地，Dynamo 跳过这一帧就抛：

```
RuntimeError: torch.compile with fullgraph=True found no compiled frames
  - _rms_norm_preserve_dtype (norm.py:15): Dynamo decided to skip the frame: BF16 is not supported
```

下面这个 cell 打两个补丁，都只改一行、都能一键还原：

| 文件 | 改动 | 为什么 |
| --- | --- | --- |
| `nn/modules/norm.py` | `torch.compile(...)` → 原始函数（eager） | T4 生成不了 bf16 内核，`fullgraph=True` 没有降级余地 |
| `nn/modules/pair_bias_attn.py` | `if not torch.cuda.is_available():` → `if True:` | 强制走文件里现成的 einsum 分支 |

⚠️ 改完**必须同时用 `precision=32-true`**：默认 `bf16-mixed` 在 T4 上不止编译不了，cuBLAS 的 bf16 GEMM 在 sm_75 也没有原生支持。

In [4]:
import pathlib

ROOT = pathlib.Path('/content/RFD4-Proteina/src/rfproteina')

# 补丁 1：RMSNorm 的 torch.compile -> eager（T4 无 bf16 inductor 支持，fullgraph=True 会直接抛错）
NORM = ROOT / 'nn/modules/norm.py'
norm_needle = '_compiled_rms_norm = torch.compile(_rms_norm_preserve_dtype, fullgraph=True, dynamic=True)'
norm_mark = '_compiled_rms_norm = _rms_norm_preserve_dtype  # [colab-patch] T4 no bf16 inductor: use eager'

# 补丁 2：注意力走文件里现成的 einsum 分支，绕开 SDPA 调度
ATTN = ROOT / 'nn/modules/pair_bias_attn.py'
attn_needle = '        if not torch.cuda.is_available():'
attn_mark = '        if True:  # [colab-patch] force einsum, bypass SDPA dispatch'

# 判重只看 '[colab-patch]' 标记，不看具体措辞：标记文本改过之后，
# 仍然能认出自己早先打下的补丁（否则会误报 "needle appears 0x"）。
for path, needle, mark, label in (
    (NORM, norm_needle, norm_mark, 'RMSNorm compile -> eager'),
    (ATTN, attn_needle, attn_mark, 'attention SDPA -> einsum'),
):
    if not path.exists():
        print(f'[skip] {path} not found')
        continue
    src = path.read_text(encoding='utf-8')
    if '[colab-patch]' in src:
        print(f'[already patched] {label}')
    elif src.count(needle) == 1:
        path.write_text(src.replace(needle, mark, 1), encoding='utf-8')
        print(f'[patched] {label}  ->  {path}')
    else:
        print(f'[skip] {label}: needle appears {src.count(needle)}x (expected 1), left untouched')

print()
print('确认补丁是否都在：')
print("  grep -n 'colab-patch' src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py")
print('还原 / revert:')
print('  cd /content/RFD4-Proteina && git checkout -- src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py')
print('然后务必用 precision=32-true 跑（见下一个 cell）。')

[already patched] RMSNorm compile -> eager
[already patched] attention SDPA -> einsum

确认补丁是否都在：
  grep -n 'colab-patch' src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py
还原 / revert:
  cd /content/RFD4-Proteina && git checkout -- src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py
然后务必用 precision=32-true 跑（见下一个 cell）。


## 跑 smoke（fp32）

`precision=32-true` 是必须的，`num_workers=2` 是为了消掉 Colab 上 DataLoader worker 过多的警告。产物在 `runs/smoke-t4/inference_samples/`。

In [5]:
%%bash
cd /content/RFD4-Proteina
set -e
unset MPLBACKEND 2>/dev/null || true

# T4 专用：bf16 在 sm_75 上没有原生支持，所以强制 fp32 + 少开 worker。
# 前提：上面 sdpa-patch 的两个补丁已生效。
.venv/bin/rfd4-generate inputs=data/benchmarks/monomer subset_to_examples=monomer-short \
    num_replicates=1 out_dir=./runs/smoke-t4 compile_lengths=null \
    precision=32-true num_workers=2

echo '--- 产物 ---'
find runs/smoke-t4 -name '*.cif*' | head -20
find runs/smoke-t4 -name '*.cif*' | wc -l

sizzling... 🔥 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:03 • 0:00:00 0.00it/s 
[2026-11-17 10:13:15,488][rfproteina.generate][INFO] - [rank: 0] ✓ Generation complete! Results saved to ./runs/smoke-t4
--- 产物 ---
runs/smoke-t4/inference_samples/monomer-short/monomer-short_generated.cif
1


Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
Environment variable PDB_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository
/content/RFD4-Proteina/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 418.57it/s]
Environment variable

In [ ]:
!pwd

/content


## Motif scaffolding：固定几个残基的原子，让模型补出脚手架

核心只有三件事，缺一件就会得到意料之外的结果（RFD4 的原则是**没写就等于没给**）：

| 要做的事 | 写在 spec 的哪里 |
| --- | --- |
| 指出哪段是 motif、哪段要生成 | `contig`：来自输入的片段用链字母（`A30-60`），要生成的长度用纯数字（`20-40`）或带名字的 `N:20-40` |
| 固定 motif 的原子 | `C_CRD`（**绝对坐标固定**）或 `C_CRDGRP`（**相对几何固定、整体可自由摆位**） |
| 是否保留 motif 的序列 | `C_SEQ: {true: [...]}`（不写 = 序列交给模型重设计） |

三者的区别很关键：

- **`C_CRD`**：把这些原子钉在原位姿上，适合「就要这个 motif 在这套坐标下」。
- **`C_CRDGRP`**：给一组原子同一个正数 group id，只保证组内相对几何，整组可以平移旋转。适合「motif 要刚性、但允许它浮到别处」。用它时**不要**在同一批原子上再写 `C_CRD`。
- **`C_IDX`**：只固定序列顺序、不在乎具体编号时用（unindexed motif）。

`input_as_annotated_cif` 也要选对：

- 自己带来的 PDB/CIF（只含 motif，可能是原始实验结构）→ `false`，让工具补全缺失原子、`MSE`→`MET`、清掉旧标注；
- 仓库里**已经标注过条件**的 CIF（M-CSA 基准、你之前导出的 annotated CIF）→ `true`，保留已有条件。

流程建议：**先 dry-run 物化成 CIF 核对，再花 GPU 跑**。`scripts/write_condition_spec_cifs.py` 生成的就是 `rfd4-generate` 实际会读的那份结构，选择串选不到原子之类的问题在这里就会暴露。

In [ ]:
import gzip
import io
import pathlib
import sys

CIF_PATH = "/content/RFD4-Proteina/data/benchmarks/mcsa/mcsa_37/M0907_1rbl.cif.zst"
WANT_CHAINS = None          # 例如 ["A"]；None = 全部
MAX_RESIDUES = 80           # 打印多少个残基（避免刷屏）

# 想固定主链就列进这里；想连侧链一起固定就留空字符串
BACKBONE_ATOMS = ("N", "CA", "C", "O")


def read_bytes(path):
    p = pathlib.Path(path)
    raw = p.read_bytes()
    if p.suffix == ".zst":
        import zstandard as zstd
        with zstd.ZstdDecompressor().stream_reader(io.BytesIO(raw)) as r:
            return r.read()
    if p.suffix == ".gz":
        return gzip.decompress(raw)
    return raw


def split_line(line):
    """按空白切分，尊重单/双引号。"""
    out, cur, quote = [], [], None
    for ch in line:
        if quote:
            if ch == quote:
                quote = None
            else:
                cur.append(ch)
        elif ch in "'\"":
            quote = ch
        elif ch.isspace():
            if cur:
                out.append("".join(cur)); cur = []
        else:
            cur.append(ch)
    if cur:
        out.append("".join(cur))
    return out


def atom_site_rows(text):
    lines = text.splitlines()
    cols, rows, i = None, [], 0
    while i < len(lines):
        ln = lines[i].strip()
        if ln == "loop_":
            j, header = i + 1, []
            while j < len(lines) and lines[j].strip().startswith("_"):
                header.append(lines[j].strip())
                j += 1
            if header and header[0].startswith("_atom_site."):
                cols = [h.split(".", 1)[1] for h in header]
                k = j
                while k < len(lines):
                    inter = lines[k].strip()
                    if not inter or inter.startswith("#"):
                        if rows:
                            break
                        k += 1
                        continue
                    if inter.startswith("_") or inter == "loop_":
                        break
                    vals = split_line(inter)
                    if vals and len(vals) >= len(cols):
                        rows.append(vals[: len(cols)])
                    k += 1
                break
            i = j
        else:
            i += 1
    return cols, rows


text = read_bytes(CIF_PATH).decode("utf-8", "replace")
cols, rows = atom_site_rows(text)
if not cols:
    sys.exit(f"没在 {CIF_PATH} 里找到 _atom_site 循环")


def pick(*names, default=""):
    for n in names:
        if n in cols:
            return cols.index(n)
    return None


idx_chain = pick("auth_asym_id", "label_asym_id")
idx_seq = pick("auth_seq_id", "label_seq_id")
idx_comp = pick("auth_comp_id", "label_comp_id")
idx_atom = pick("auth_atom_id", "label_atom_id")
idx_model = pick("pdbx_PDB_model_num")

groups = {}
for r in rows:
    if idx_model is not None and r[idx_model] not in ("1", "."):
        continue
    chain = r[idx_chain]
    if WANT_CHAINS and chain not in WANT_CHAINS:
        continue
    seq = r[idx_seq]
    key = (chain, seq)
    entry = groups.setdefault(key, {"res": r[idx_comp], "atoms": [], "names": []})
    entry["names"].append(r[idx_atom])
    entry["atoms"].append(r[idx_atom])

print(f"{CIF_PATH}")
print(f"  链: {sorted({k[0] for k in groups})}   残基总数: {len(groups)}")
print()
print(f"{'chain':>5} {'res_id':>7} {'res_name':>8}  atoms")
shown = 0
for (chain, seq), info in groups.items():
    if shown >= MAX_RESIDUES:
        print(f"  ... 还有 {len(groups) - shown} 个残基未打印（调大 MAX_RESIDUES）")
        break
    print(f"{chain:>5} {seq:>7} {info['res']:>8}  {','.join(info['names'])}")
    shown += 1

# 生成可直接粘贴的选择串：把同一残基里需要的原子挑出来
print()
print("=" * 70)
print("主链固定的选择串（复制到 C_CRD 的 select 里）：")
for (chain, seq), info in list(groups.items())[:5]:
    have = [a for a in BACKBONE_ATOMS if a in info["names"]]
    if have:
        print(f"  sel('{chain}/{info['res']}/{seq}/[{','.join(have)}]')")
print()
print("整条链（所有原子）：sel('A')  |  某个残基全部原子：sel('A/LEU/45')")
print("只要主链、不碰侧链：  sel('A') and is_protein_backbone")


In [ ]:
import pathlib

ROOT = pathlib.Path("/content/RFD4-Proteina")

# ① 开箱即用：仓库自带的 M-CSA 溶菌酶位点（催化残基 + 配体 + 脚手架长度都已标注）
(ROOT / "motif.yaml").write_text(
    """rubisco_site:
  input: data/benchmarks/mcsa/mcsa_37/M0907_1rbl.cif.zst
  input_as_annotated_cif: true
""",
    encoding="utf-8",
)

# ② 你自己的 motif：改 input / contig / conditions 三处
CUSTOM = """my_motif:
  # 你的结构文件；相对路径按「本 YAML 所在目录」解析
  input: /content/RFD4-Proteina/path/to/your_motif.cif
  # 原始结构 -> false（补全原子、MSE->MET、清掉旧标注）
  input_as_annotated_cif: false

  # contig = 来自输入的片段 + 要生成的长度
  #   A30-60   = 输入的 A 链 30..60
  #   N:20-40  = 长度在 20..40 随机，名字叫 N（方便后面 select）
  #   /0       = 链断裂
  # 至少保留一段来自 input 的片段，否则会被拒绝
  contig: "A30-60, N:20-40, C:20-40"

  conditions:
    # 把 motif 原子钉死在原坐标（要整体可动就换成 C_CRDGRP）
    C_CRD:
      true:
        - select: "sel('A/ALA/45/[N,CA,C,O,CB]')"
        - select: "sel('A/LEU/46/[N,CA,C,O]')"
    # 保留 motif 的序列身份（不写这段，motif 的序列会被重新设计）
    C_SEQ:
      true:
        - select: "sel('A/ALA/45')"
        - select: "sel('A/LEU/46')"

    # 想要「刚性但可自由摆位」就用这个替代 C_CRD（组号用同一个正数）：
    # C_CRDGRP:
    #   1:
    #     - select: "sel('A/ALA/45/[N,CA,C,O,CB]')"
    #     - select: "sel('A/LEU/46/[N,CA,C,O]')"

    # 核酸 motif 的原子名不同（碱基环 N9/C8/N7/C5/C6/N1/C2/N3/C4，磷酸-糖 P/O5'/C5'/C4'/C3'/O3'/C2'/C1'）：
    # C_CRD:
    #   true:
    #     - select: "sel('B/DA/5/[C2,N3,C4,C5,C6,N1]')"   # 只固定碱基环
    #     - select: "sel('B/DA/5')"                       # 整个核苷酸
"""
(ROOT / "motif_custom.yaml").write_text(CUSTOM, encoding="utf-8")

for name in ("motif.yaml", "motif_custom.yaml"):
    p = ROOT / name
    print(f"已写入 {p}（{len(p.read_text(encoding='utf-8').splitlines())} 行）")


In [ ]:
%%bash
cd /content/RFD4-Proteina
set -e
unset MPLBACKEND 2>/dev/null || true

# 物化成 annotated CIF —— 这就是 rfd4-generate 实际会读的结构。
# 选择串选不到原子、contig 没有保留片段之类的问题会在这里直接报出来，不浪费 GPU。
.venv/bin/python scripts/write_condition_spec_cifs.py motif.yaml --out-dir ./cifs

echo '--- 产物 ---'
find cifs -name '*.cif*' | sed 's/^/  /'
echo '--- 把这个 CIF 下载下来在 PyMOL / 浏览器里核对 motif 位置 ---'


In [ ]:
%%bash
cd /content/RFD4-Proteina
set -e
unset MPLBACKEND 2>/dev/null || true

# T4 专用：fp32（bf16 在 sm_75 上没有原生支持），前提是 sdpa-patch 的两个补丁已生效。
# 显存只有 15GB，先把 batch_size 压到 1、num_replicates 降到 2。
.venv/bin/rfd4-generate inputs=motif.yaml recycle_strategy=medium/midpeak \
    num_replicates=2 batch_size=1 out_dir=./runs/motif \
    prevalidate_inputs=true '~metrics' \
    precision=32-true num_workers=2

echo '--- 产物 ---'
find runs/motif -name '*.cif*' | head -10
find runs/motif -name '*.cif*' | wc -l


## 把产物拿到本地

产物在 `runs/<name>/inference_samples/`（结构 CIF）和 `inference_metrics/`（指标 JSON，只有在跑了 metrics 时才有）。**`/content` 是临时的，会话一断整个目录就没了**——先导出，再慢慢看。

两种方式：

- **A. 直接下载**（下面第一个 cell）：把目录压成一个 zip 交给浏览器下载。几十 MB 很稳；超过几百 MB 时浏览器下载容易断。
- **B. 走 Google Drive**（下面第二个 cell）：挂载 Drive，再把目录拷进去，然后从 Drive 网页或桌面版同步到本地。适合大的、或者要反复取的。

> 关于让模型**直接写进 Drive**（`out_dir=/content/drive/MyDrive/...`）：Drive 的 FUSE 写入很慢，模型又是高频小文件写入，容易把一次跑不动的时间拖长好几倍。所以推荐“先在 `/content` 生成、最后一次性拷走”。

> 想把 1.7GB 权重缓存也省掉重下：`export HF_HOME=/content/drive/MyDrive/hf_cache`（同样会慢一些，权衡着用）。

In [12]:
import os
import shutil
import time

from google.colab import files  # 只在 Colab 运行时里存在

# 要导出的目录。整个 runs/ 一起打包也可以：
#   RUN_DIR = "/content/RFD4-Proteina/runs"
RUN_DIR = "/content/RFD4-Proteina/runs/smoke-t4"

stamp = time.strftime("%Y%m%d-%H%M%S")
zip_base = f"/content/{os.path.basename(RUN_DIR)}-{stamp}"
zip_path = shutil.make_archive(zip_base, "zip", RUN_DIR)
size_mb = os.path.getsize(zip_path) / 1e6
print(f"打包完成：{zip_path}（{size_mb:.1f} MB）")

if size_mb < 300:
    files.download(zip_path)   # 浏览器会弹下载
else:
    print("超过 300MB，浏览器下载容易失败 —— 用下面那个 Google Drive cell。")

打包完成：/content/smoke-t4-20261117-102332.zip（0.0 MB）


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
import os
import shutil
import time

from google.colab import drive

# 同上，可以改成整个 runs/
RUN_DIR = "/content/RFD4-Proteina/runs/smoke-t4"

drive.mount("/content/drive")            # 会弹授权
stamp = time.strftime("%Y%m%d-%H%M%S")
dest = f"/content/drive/MyDrive/rfd4-runs/{os.path.basename(RUN_DIR)}-{stamp}"
os.makedirs(os.path.dirname(dest), exist_ok=True)
shutil.copytree(RUN_DIR, dest)
print("已拷到 Drive：", dest)
print("然后可以在 Drive 网页端下载，或用 Google Drive 桌面版同步到本地。")

Mounted at /content/drive
已拷到 Drive： /content/drive/MyDrive/rfd4-runs/smoke-t4-20261117-102119
然后可以在 Drive 网页端下载，或用 Google Drive 桌面版同步到本地。


## T4 上的注意事项

- **慢、费显存**：跑的是 fp32 + eager einsum，不是 bf16 + 融合内核。`monomer-short` 一个设计约 30 秒；稍大的输入很可能 15GB OOM。显存紧张时同时降 `batch_size=1` 和 `num_replicates=1`。
- **必须带 `precision=32-true`**：不加就回到 `bf16-mixed`，会在 `torch.compile` 或 cuBLAS 的 bf16 GEMM 上再次失败。
- **补丁必须已生效**：`sdpa-patch` cell 会打印 `[patched]` 或 `[already patched]`。确认方法：`grep -n 'colab-patch' src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py`
- **还原**（换回 L4/A100 前一定要做，否则白丢 bf16 加速）：`cd /content/RFD4-Proteina && git checkout -- src/rfproteina/nn/modules/norm.py src/rfproteina/nn/modules/pair_bias_attn.py`
- `CCD_MIRROR_PATH / PDB_MIRROR_PATH not set`：atomworks 的提示，只影响需要本地镜像的功能，可忽略。
- `ValueError: Key backend: 'module://matplotlib_inline...'`：`MPLBACKEND` 没清掉，回上面第 4 个 cell。
- `operator torchvision::nms does not exist`：跑的是 `/usr/local/bin/rfd4-generate`（系统环境里的旧安装），必须用 `.venv/bin/rfd4-generate`。
- 有 L4/A100 额度时**优先用 `RFD4-colab.ipynb`**：同样的代码路径，bf16 有硬件加速、显存也宽裕。